In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
%cd /content

!rm -rf /content/smallnet
!git clone https://github.com/SepehrAkbari/smallnet.git

%cd /content/smallnet

!git status --short
!git log -1 --oneline

/content
Cloning into 'smallnet'...
remote: Enumerating objects: 2180, done.
remote: Counting objects: 100% (295/295), done.
remote: Compressing objects: 100% (219/219), done.
remote: Total 2180 (delta 173), reused 186 (delta 73), pack-reused 1885 (from 1)
Receiving objects: 100% (2180/2180), 582.53 MiB | 20.41 MiB/s, done.
Resolving deltas: 100% (408/408), done.
Updating files: 100% (1666/1666), done.
/content/smallnet
de476fb (HEAD -> main, origin/main, origin/HEAD) Update n5.ipynb


In [ ]:
!uv --version
!uv sync

uv 0.11.19 (x86_64-unknown-linux-gnu)
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 85 packages in 17ms
Prepared 79 packages in 35.57s                                           
Installed 79 packages in 259ms                              
 + asttokens==3.0.1
 + comm==0.2.3
 + contourpy==1.3.3
 + cuda-bindings==13.2.0
 + cuda-pathfinder==1.5.4
 + cuda-toolkit==13.0.2
 + cycler==0.12.1
 + debugpy==1.8.20
 + decorator==5.3.1
 + executing==2.2.1
 + filelock==3.29.0
 + fonttools==4.63.0
 + fsspec==2026.4.0
 + iniconfig==2.3.0
 + ipykernel==7.2.0
 + ipython==9.13.0
 + ipython-pygments-lexers==1.1.1
 + ipywidgets==8.1.8
 + jedi==0.20.0
 + jinja2==3.1.6
 + jupyter-client==8.8.0
 + jupyter-core==5.9.1
 + jupyterlab-widgets==3.0.16
 + kiwisolver==1.5.0
 + markupsafe==3.0.3
 + matplotlib==3.10.9
 + matplotlib-inline==0.2.2
 + mpmath==1.3.0
 + nest-asyncio==1.6.0
 + networkx==3.6.1
 + numpy==2.4.4
 + nvidia-cublas==13.1.1.3
 + nvidia-cuda-cu

In [ ]:
from pathlib import Path
import subprocess

repo = Path("/content/smallnet")
backup = Path(
    "/content/drive/MyDrive/smallnet_colab_backup"
)

assert backup.exists(), f"Backup directory not found: {backup}"

restore_pairs = [
    (
        backup / "camvid_vgg_cp",
        repo / "results/camvid_vgg_cp",
    ),
    (
        backup / "paper",
        repo / "results/paper",
    ),
]

for source, destination in restore_pairs:
    if source.exists():
        destination.mkdir(parents=True, exist_ok=True)

        subprocess.run(
            [
                "rsync",
                "-a",
                f"{source}/",
                f"{destination}/",
            ],
            check=True,
        )

        print(f"Restored: {source}")
    else:
        print(f"Not found, skipped: {source}")

# Prevent the old accidental duplicate nesting from returning.
duplicate = (
    repo
    / "results/camvid_vgg_cp/camvid_vgg_cp"
)

if duplicate.exists():
    subprocess.run(
        ["rm", "-rf", str(duplicate)],
        check=True,
    )
    print("Removed accidental duplicate nesting.")
    
from pathlib import Path
import shutil

duplicate_backup = Path(
    "/content/drive/MyDrive/"
    "smallnet_colab_backup/"
    "camvid_vgg_cp/camvid_vgg_cp"
)

if duplicate_backup.exists():
    shutil.rmtree(duplicate_backup)
    print("Removed duplicate nesting from Drive backup.")
else:
    print("No duplicate Drive directory found.")
    
from pathlib import Path
import shutil
import subprocess

drive_root = Path("/content/drive/MyDrive")
repo = Path("/content/smallnet")

model_candidates = [
    drive_root
    / "smallnet_colab_backup/model/best_model.pth",
    drive_root
    / "smallnet/model/best_model.pth",
]

model_source = next(
    (
        path
        for path in model_candidates
        if path.is_file()
    ),
    None,
)

if model_source is None:
    model_matches = list(
        drive_root.rglob("best_model.pth")
    )
    model_source = (
        model_matches[0]
        if model_matches
        else None
    )

assert model_source is not None, (
    "Could not find best_model.pth in Google Drive."
)

model_destination = (
    repo / "model/best_model.pth"
)
model_destination.parent.mkdir(
    parents=True,
    exist_ok=True,
)
shutil.copy2(
    model_source,
    model_destination,
)

print("Model source:", model_source)
print("Model copied to:", model_destination)

from pathlib import Path

repo = Path("/content/smallnet")

required = [
    repo / "model/best_model.pth",
    repo / "data/CamVid/class_dict.csv",
    repo / "data/CamVid/train",
    repo / "data/CamVid/train_labels",
    repo / "data/CamVid/val",
    repo / "data/CamVid/val_labels",
    repo / "data/CamVid/test",
    repo / "data/CamVid/test_labels",
    repo
    / "results/camvid_vgg_cp/"
    "dataset_validation_report.json",
]

for path in required:
    print(
        "OK" if path.exists() else "MISSING",
        path,
    )

assert all(path.exists() for path in required)

from pathlib import Path

camvid = Path("/content/smallnet/data/CamVid")

for split in ["train", "val", "test"]:
    image_count = len(
        list((camvid / split).glob("*"))
    )
    mask_count = len(
        list(
            (
                camvid / f"{split}_labels"
            ).glob("*")
        )
    )

    print(
        split,
        "images:",
        image_count,
        "masks:",
        mask_count,
    )

Restored: /content/drive/MyDrive/smallnet_colab_backup/camvid_vgg_cp
Restored: /content/drive/MyDrive/smallnet_colab_backup/paper
No duplicate Drive directory found.
Model source: /content/drive/MyDrive/model/best_model.pth
Model copied to: /content/smallnet/model/best_model.pth
OK /content/smallnet/model/best_model.pth
OK /content/smallnet/data/CamVid/class_dict.csv
OK /content/smallnet/data/CamVid/train
OK /content/smallnet/data/CamVid/train_labels
OK /content/smallnet/data/CamVid/val
OK /content/smallnet/data/CamVid/val_labels
OK /content/smallnet/data/CamVid/test
OK /content/smallnet/data/CamVid/test_labels
OK /content/smallnet/results/camvid_vgg_cp/dataset_validation_report.json
train images: 369 masks: 369
val images: 100 masks: 100
test images: 232 masks: 232


In [ ]:
!nvidia-smi

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )

assert torch.cuda.is_available()

Sat Jul 25 18:17:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import json
from pathlib import Path

config_path = Path(
    "/content/smallnet/"
    "configs/camvid_vgg_cp_paper.json"
)

config = json.loads(
    config_path.read_text()
)

print(
    json.dumps(
        config.get("final_structural", {}),
        indent=2,
    )
)

{
  "ranks": [
    32,
    64,
    128,
    256,
    512
  ],
  "seeds": [
    0,
    1,
    2
  ],
  "iteration_budget": 200,
  "init": "random",
  "memory_efficient_mttkrp": true,
  "mttkrp_rank_chunk_size": 64,
  "mttkrp_max_explicit_bytes": 536870912,
  "numerical_tolerance": 1e-05,
  "residual_output_chunk_size": 8,
  "factor_diagnostic_thresholds": {
    "near_zero_component_norm_threshold": 1e-12,
    "extreme_factor_norm_threshold": 1000000.0,
    "scaling_spread_threshold": 1000000.0,
    "cancellation_ratio_threshold": 1000.0,
    "bounded_reconstruction_multiple": 10.0
  },
  "output_dir": "results/camvid_vgg_cp/final_structural",
  "figures_dir": "results/paper/figures",
  "audit_path": "results/paper/final_structural_audit.md"
}


In [ ]:
%cd /content/smallnet

!uv run python -m pytest -q

/content/smallnet
..................................................s..................... [ 78%]
....................                                                     [100%]
=============================== warnings summary ===============================
.venv/lib/python3.12/site-packages/tensorly/solvers/nnls.py:103
  /content/smallnet/.venv/lib/python3.12/site-packages/tensorly/solvers/nnls.py:103: SyntaxWarning: invalid escape sequence '\l'
    .. math:: \lambda_s, \lambda_r

.venv/lib/python3.12/site-packages/tensorly/solvers/admm.py:99
  /content/smallnet/.venv/lib/python3.12/site-packages/tensorly/solvers/admm.py:99: SyntaxWarning: invalid escape sequence '\_'
    .. math:: dual\_var = dual\_var + (Ax + Bx_{split} - c)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
91 passed, 1 skipped, 2 warnings in 53.86s


In [ ]:
from pathlib import Path
import subprocess

REPO = Path("/content/smallnet")
BACKUP = Path(
    "/content/drive/MyDrive/"
    "smallnet_colab_backup"
)

def backup_results():
    pairs = [
        (
            REPO / "results/camvid_vgg_cp",
            BACKUP / "camvid_vgg_cp",
        ),
        (
            REPO / "results/paper",
            BACKUP / "paper",
        ),
    ]

    for source, destination in pairs:
        destination.mkdir(
            parents=True,
            exist_ok=True,
        )

        subprocess.run(
            [
                "rsync",
                "-a",
                f"{source}/",
                f"{destination}/",
            ],
            check=True,
        )

    duplicate = (
        BACKUP
        / "camvid_vgg_cp/camvid_vgg_cp"
    )

    if duplicate.exists():
        subprocess.run(
            ["rm", "-rf", str(duplicate)],
            check=True,
        )

    print("Results backed up to Google Drive.")

In [ ]:
backup_results()

Results backed up to Google Drive.


In [ ]:
import csv
import subprocess
from pathlib import Path

def show_progress():
    summary_path = (
        REPO
        / "results/camvid_vgg_cp/"
        "final_structural/"
        "final_structural_summary.csv"
    )

    if not summary_path.exists():
        print("No summary written yet.")
        return

    with summary_path.open(newline="") as file:
        rows = list(csv.DictReader(file))

    completed = [
        row
        for row in rows
        if row.get("status") == "completed"
    ]

    cp_rows = [
        row
        for row in completed
        if row.get("method")
        == "cp_200_iterations"
    ]

    svd_rows = [
        row
        for row in completed
        if row.get("method")
        == "matrix_svd_output_unfolding"
    ]

    print("Total rows:", len(rows))
    print("Completed rows:", len(completed))
    print("Completed CP rows:", len(cp_rows))
    print("Completed SVD rows:", len(svd_rows))

    completed_by_rank = {}

    for row in completed:
        rank = row.get("rank")
        completed_by_rank.setdefault(
            rank,
            [],
        ).append(
            (
                row.get("method"),
                row.get("seed"),
            )
        )

    for rank in sorted(
        completed_by_rank,
        key=int,
    ):
        print(
            f"Rank {rank}:",
            completed_by_rank[rank],
        )


def run_final_structural_rank(rank):
    command = [
        "uv",
        "run",
        "python",
        "scripts/run_experiment.py",
        "--config",
        "configs/camvid_vgg_cp_paper.json",
        "--stage",
        "final-structural",
        "--device",
        "cuda",
        "--ranks",
        str(rank),
        "--seeds",
        "0",
        "1",
        "2",
    ]

    print(
        "Running:",
        " ".join(command),
    )

    try:
        subprocess.run(
            command,
            cwd=REPO,
            check=True,
        )
    finally:
        backup_results()
        show_progress()

In [ ]:
run_final_structural_rank(32)
show_progress()
backup_results()
run_final_structural_rank(64)
show_progress()
backup_results()
run_final_structural_rank(128)
show_progress()
backup_results()
run_final_structural_rank(256)
show_progress()
backup_results()
run_final_structural_rank(512)
show_progress()
backup_results()

Running: uv run python scripts/run_experiment.py --config configs/camvid_vgg_cp_paper.json --stage final-structural --device cuda --ranks 32 --seeds 0 1 2
Results backed up to Google Drive.
Total rows: 4
Completed rows: 3
Completed CP rows: 3
Completed SVD rows: 0
Rank 32: [('cp_200_iterations', '0'), ('cp_200_iterations', '1'), ('cp_200_iterations', '2')]
Total rows: 4
Completed rows: 3
Completed CP rows: 3
Completed SVD rows: 0
Rank 32: [('cp_200_iterations', '0'), ('cp_200_iterations', '1'), ('cp_200_iterations', '2')]
Results backed up to Google Drive.
Running: uv run python scripts/run_experiment.py --config configs/camvid_vgg_cp_paper.json --stage final-structural --device cuda --ranks 64 --seeds 0 1 2
Results backed up to Google Drive.
Total rows: 8
Completed rows: 6
Completed CP rows: 6
Completed SVD rows: 0
Rank 32: [('cp_200_iterations', '0'), ('cp_200_iterations', '1'), ('cp_200_iterations', '2')]
Rank 64: [('cp_200_iterations', '0'), ('cp_200_iterations', '1'), ('cp_200_ite

In [ ]:
%cd /content/smallnet

import csv
import hashlib
import json
import math
from pathlib import Path

root = Path(
    "results/camvid_vgg_cp/final_structural"
)

summary_path = root / "final_structural_summary.csv"

assert summary_path.is_file(), summary_path

rows = list(csv.DictReader(summary_path.open()))

cp = [
    row
    for row in rows
    if row["method"] == "cp_200_iterations"
]

expected_ranks = {32, 64, 128, 256, 512}
expected_seeds = {0, 1, 2}

cp_keys = {
    (
        int(float(row["rank"])),
        int(float(row["seed"])),
        int(float(row["iteration_budget"])),
    )
    for row in cp
}

expected_keys = {
    (rank, seed, 200)
    for rank in expected_ranks
    for seed in expected_seeds
}

assert cp_keys == expected_keys, {
    "missing": sorted(expected_keys - cp_keys),
    "unexpected": sorted(cp_keys - expected_keys),
}

assert len(cp) == 15
assert all(row["status"] == "completed" for row in cp)

numeric_fields = [
    "actual_relative_squared_frobenius_error",
    "actual_relative_frobenius_error",
    "activation_normalized_squared_error",
    "activation_relative_frobenius_error",
    "activation_cosine_similarity",
    "activation_absolute_mse",
    "validation_present_class_miou",
    "test_present_class_miou",
    "target_layer_parameter_count",
    "full_model_parameter_count",
    "target_layer_macs",
    "full_model_macs",
    "compression_factor",
]

for row in cp:
    for field in numeric_fields:
        value = float(row[field])
        assert math.isfinite(value), (
            row["rank"],
            row["seed"],
            field,
            value,
        )

assert len({
    row["checkpoint_sha256"]
    for row in cp
}) == 1

assert len({
    row["dataset_validation_report_sha256"]
    for row in cp
}) == 1

assert len({
    row["target_tensor_sha256"]
    for row in cp
}) == 1

for row in cp:
    assert row["iteration_budget"] == "200"

    assert (
        row["same_fitted_factors_reused_for_all_metrics"]
        == "True"
    )

    factor_hashes = {
        row["final_factor_hash_sha256"],
        row["factor_hash_after_reconstruction"],
        row["factor_hash_after_activation_distortion"],
        row["factor_hash_after_zero_shot"],
    }

    assert len(factor_hashes) == 1

    artifact = Path(row["factor_artifact_path"])

    assert artifact.is_file(), artifact

    actual_artifact_hash = hashlib.sha256(
        artifact.read_bytes()
    ).hexdigest()

    assert (
        actual_artifact_hash
        == row["factor_artifact_sha256"]
    )

metadata = json.loads(
    (
        root / "final_structural_metadata.json"
    ).read_text()
)

assert metadata["iteration_budget"] == 200
assert metadata["preliminary_artifacts_preserved"]

reuse_checks = metadata[
    "same_fitted_factor_reuse_checks"
]

assert len(reuse_checks) == 15
assert all(
    check["same_fitted_factors_reused_for_all_metrics"]
    for check in reuse_checks
)

print("CP-only final structural validation: PASS")
print("Validated 15 completed CP rows and factor artifacts.")

svd_rows = [
    row
    for row in rows
    if row["method"]
    == "matrix_svd_output_unfolding"
]

print(
    f"Matrix-SVD rows retained but excluded: "
    f"{len(svd_rows)}"
)

/content/smallnet
CP-only final structural validation: PASS
Validated 15 completed CP rows and factor artifacts.
Matrix-SVD rows retained but excluded: 5


In [ ]:
import pandas as pd
from pathlib import Path

root = Path(
    "/content/smallnet/results/"
    "camvid_vgg_cp/final_structural"
)

summary = pd.read_csv(
    root / "final_structural_summary.csv"
)

cp = summary[
    (summary["method"] == "cp_200_iterations")
    & (summary["status"] == "completed")
].copy()

cp = cp.sort_values(["rank", "seed"])

cp_output = (
    root / "final_structural_cp_only_summary.csv"
)

cp.to_csv(cp_output, index=False)

rank_summary = (
    cp.groupby("rank", as_index=False)
    .agg(
        reconstruction_error_mean=(
            "actual_relative_squared_frobenius_error",
            "mean",
        ),
        reconstruction_error_std=(
            "actual_relative_squared_frobenius_error",
            "std",
        ),
        activation_error_mean=(
            "activation_normalized_squared_error",
            "mean",
        ),
        activation_error_std=(
            "activation_normalized_squared_error",
            "std",
        ),
        validation_miou_mean=(
            "validation_present_class_miou",
            "mean",
        ),
        validation_miou_std=(
            "validation_present_class_miou",
            "std",
        ),
        test_miou_mean=(
            "test_present_class_miou",
            "mean",
        ),
        test_miou_std=(
            "test_present_class_miou",
            "std",
        ),
        compression_factor=(
            "compression_factor",
            "first",
        ),
        target_layer_parameter_count=(
            "target_layer_parameter_count",
            "first",
        ),
        full_model_parameter_count=(
            "full_model_parameter_count",
            "first",
        ),
        target_layer_macs=(
            "target_layer_macs",
            "first",
        ),
        full_model_macs=(
            "full_model_macs",
            "first",
        ),
    )
)

rank_output = (
    root / "final_structural_cp_only_rank_summary.csv"
)

rank_summary.to_csv(rank_output, index=False)

display(rank_summary)

print("Saved:", cp_output)
print("Saved:", rank_output)

,rank,reconstruction_error_mean,reconstruction_error_std,activation_error_mean,activation_error_std,validation_miou_mean,validation_miou_std,test_miou_mean,test_miou_std,compression_factor,target_layer_parameter_count,full_model_parameter_count,target_layer_macs,full_model_macs
0,32,0.944835,0.000125,0.332907,0.001448,0.220626,0.003447,0.202023,0.002907,675.940223,152032.0,31779136.0,24409440.0,5.449171e+10
1,64,0.923266,0.000342,0.295760,0.000998,0.249025,0.001019,0.225684,0.002915,342.585022,299968.0,31927072.0,48818880.0,5.451612e+10
2,128,0.891966,0.000164,0.258819,0.000719,0.274627,0.002510,0.249013,0.001865,172.470032,595840.0,32222944.0,97637760.0,5.456494e+10
3,256,0.846052,0.000088,0.218419,0.000067,0.293858,0.000699,0.269925,0.001077,86.532442,1187584.0,32814688.0,195275520.0,5.466257e+10
4,512,0.780846,0.000109,0.174798,0.000303,0.309271,0.001162,0.287181,0.000322,43.340963,2371072.0,33998176.0,390551040.0,5.485785e+10


Saved: /content/smallnet/results/camvid_vgg_cp/final_structural/final_structural_cp_only_summary.csv
Saved: /content/smallnet/results/camvid_vgg_cp/final_structural/final_structural_cp_only_rank_summary.csv


In [ ]:
backup_results()

Results backed up to Google Drive.


In [ ]:
!pwd

/content/smallnet


In [ ]:
# !cd /Users/sepehrakbari/Projects/smallnet

# rsync -av \
#   '/Users/sepehrakbari/Library/CloudStorage/GoogleDrive-isepehrakbari@gmail.com/My Drive/smallnet_colab_backup/camvid_vgg_cp' \
#   results/camvid_vgg_cp/

# rsync -av \
#   '/Users/sepehrakbari/Library/CloudStorage/GoogleDrive-isepehrakbari@gmail.com/My Drive/smallnet_colab_backup/paper' \
#   results/paper/

/bin/bash: line 1: cd: /Users/sepehrakbari/Projects/smallnet: No such file or directory
sending incremental file list
rsync: [sender] change_dir "/Users/sepehrakbari/Library/CloudStorage/GoogleDrive-isepehrakbari@gmail.com/My Drive/smallnet_colab_backup" failed: No such file or directory (2)

sent 19 bytes  received 12 bytes  62.00 bytes/sec
total size is 0  speedup is 0.00
rsync error: some files/attrs were not transferred (see previous errors) (code 23) at main.c(1347) [sender=3.2.7]
sending incremental file list
rsync: [sender] change_dir "/Users/sepehrakbari/Library/CloudStorage/GoogleDrive-isepehrakbari@gmail.com/My Drive/smallnet_colab_backup" failed: No such file or directory (2)

sent 19 bytes  received 12 bytes  62.00 bytes/sec
total size is 0  speedup is 0.00
rsync error: some files/attrs were not transferred (see previous errors) (code 23) at main.c(1347) [sender=3.2.7]
